# News RAG Chatbot: Hybrid Retrieval-Augmented Generation

End-to-end experiment notebook for a chatbot that answers questions about recent news, grounded in ~500 scraped articles:

1. **Scrape** ~500 articles from BBC News, The Guardian, NPR and Ars Technica (Scrapy + RSS).
2. **Baseline RAG**: sentence-level chunks → `harrier-oss-v1-0.6b` embeddings → Qdrant vector search → local LLM (Ollama) with a grounded, citation-enforcing prompt and three guardrails.
3. **Evaluate** faithfulness with DeepEval on 5 hand-written gold questions.
4. **Improve** the pipeline (2-3 sentence chunks + hybrid BM25/vector retrieval with RRF + cross-encoder reranking) and re-evaluate. Mean faithfulness went from **0.70 to 0.93**.

> The reusable version of this code (used by the Streamlit app) lives in `rag_pipeline.py` and `build_index.py`. This notebook keeps the original step-by-step experiment and its outputs.

## 1. Data collection

The goal is a chatbot that:
* retrieves the most relevant chunks from the news corpus for a user query,
* generates accurate answers grounded in the retrieved evidence,
* minimizes hallucinations and cites its sources,
* can be evaluated automatically.

The corpus is ~500 recent news articles scraped with **Scrapy**.

The spider lives in [`news_spider.py`](news_spider.py). It seeds article discovery from 34 RSS feeds (BBC, The Guardian, NPR, Ars Technica), uses a site-specific CSS-selector parser for each source to extract the article body, skips articles under 80 words, and stops at 500 articles. It respects `robots.txt` and uses AutoThrottle. Each record stores `article_id, title, url, source, topic, pub_date, description, body, word_count`. The topic is inferred from the URL path.

In [ ]:
# Run the spider from within the notebook.

import subprocess, sys

result = subprocess.run([sys.executable, "run_spider.py"], cwd=".")

if result.returncode == 0:

    print("Spider finished. Articles saved to articles.jsonl.")

else:

    print(f"Spider exited with code {result.returncode}.")

Spider finished. Articles saved to articles.jsonl.


In [20]:
import pandas as pd, json

articles = []
with open("articles.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            articles.append(json.loads(line))

df = pd.DataFrame(articles)
# Ensure article_id is an integer for use as a Qdrant payload field
df["article_id"] = df["article_id"].astype(int)
print(f"Loaded {len(df)} articles | columns: {list(df.columns)}")
df[["article_id", "title", "source", "topic", "word_count"]].head()

Loaded 500 articles | columns: ['article_id', 'title', 'url', 'source', 'topic', 'pub_date', 'description', 'body', 'word_count', 'scraped_at']


,article_id,title,source,topic,word_count
0,1,"COP30: Trump and many leaders are skipping it,...",BBC News,general,1753
1,2,The debate about whether the NHS should use ma...,BBC News,general,2015
2,3,Alex Murdaugh’s murder convictions overturned ...,The Guardian,us news,503
3,4,'I'd rather live in hiding in the US than retu...,BBC News,general,1651
4,5,The race against time to find eagles escaped f...,BBC News,general,1519


## 2. Baseline pipeline

### 2.1 Chunking: one sentence per chunk

The baseline splits every article into sentences (NLTK) and drops very short ones (< 8 words, e.g. bylines and captions). Each chunk keeps its metadata, and its `chunk_id` (`<article_id>_<sentence_index>`) is what the LLM must cite.

In [ ]:
import nltk
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize

MIN_WORDS = 8

def chunk_article(row):
    """Split one article body into sentence-level chunks with metadata."""
    chunks = []
    for i, sent in enumerate(sent_tokenize(row["body"])):
        if len(sent.split()) < MIN_WORDS:
            continue
        chunks.append({
            "chunk_id":  f"{row['article_id']}_{i}",
            "article_id": int(row["article_id"]),
            "title":     row["title"],
            "source":    row["source"],
            "topic":     row["topic"],
            "url":       row["url"],
            "text":      sent,
        })
    return chunks

all_chunks = []
for _, row in df.iterrows():
    all_chunks.extend(chunk_article(row))

chunks_df = pd.DataFrame(all_chunks)
print(f"Articles: {len(df)}  |  Sentence chunks: {len(chunks_df)}")
print(f"Avg chunks/article: {len(chunks_df)/len(df):.1f}")
chunks_df.head(3)

Articles: 500  |  Sentence chunks: 18990
Avg chunks/article: 38.0


,chunk_id,article_id,title,source,topic,url,text
0,1_0,1,"COP30: Trump and many leaders are skipping it,...",BBC News,general,https://www.bbc.com/news/articles/c205jvyg3wjo...,"There is a photograph, taken ten years ago in ..."
1,1_1,1,"COP30: Trump and many leaders are skipping it,...",BBC News,general,https://www.bbc.com/news/articles/c205jvyg3wjo...,"In it, dozens of men and women line up in dark..."
2,1_2,1,"COP30: Trump and many leaders are skipping it,...",BBC News,general,https://www.bbc.com/news/articles/c205jvyg3wjo...,Right in the middle the UK's then-Prime Minist...


### 2.2 Embeddings

Each chunk is embedded with **`microsoft/harrier-oss-v1-0.6b`** (1024-dim, normalized embeddings). The model runs on CPU or GPU.

In [22]:
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import numpy as np, torch

MODEL_ID   = "microsoft/harrier-oss-v1-0.6b"
EMBED_DIM  = 1024   # harrier-oss-v1-0.6b output dimension
BATCH_SIZE = 64     # 0.6B runs on CPU; larger batches are fine

# 0.6B model runs on CPU with float32; no GPU required.
# To use the full 27B variant swap MODEL_ID for "microsoft/harrier-oss-v1-27b" (requires GPU).
model = SentenceTransformer(
    MODEL_ID,
    
    trust_remote_code=True,
)
print(f"Loaded {MODEL_ID}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

texts = chunks_df["text"].tolist()
embeddings = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Encoding chunks"):
    batch = texts[i : i + BATCH_SIZE]
    embs  = model.encode(batch, normalize_embeddings=True, show_progress_bar=False)
    embeddings.append(embs)

embeddings = np.vstack(embeddings)
print(f"Embeddings matrix: {embeddings.shape}")

Loaded microsoft/harrier-oss-v1-0.6b
Embedding dimension: 1024


Embeddings matrix: (18990, 1024)


### 2.3 Vector store

Chunks, embeddings and metadata (title, article ID, source, topic, URL) are loaded into a **Qdrant** collection with cosine distance. Qdrant runs locally in Docker.

In [23]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

COLLECTION = "news_chunks"

client = QdrantClient(host="localhost", port=6333)

# Recreate collection so re-runs start fresh
client.recreate_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
)

# Build Qdrant point objects: id (int), vector, payload (metadata + text)
points = []
for i, (_, row) in enumerate(chunks_df.reset_index(drop=True).iterrows()):
    points.append(PointStruct(
        id      = i,
        vector  = embeddings[i].tolist(),
        payload = {
            "chunk_id":  row["chunk_id"],
            "article_id":row["article_id"],
            "title":     row["title"],
            "source":    row["source"],
            "topic":     row["topic"],
            "url":       row["url"],
            "text":      row["text"],
        },
    ))

# Upsert in batches of 256 to stay within Qdrant's request size limits
UPSERT_BATCH = 256
for start in tqdm(range(0, len(points), UPSERT_BATCH), desc="Upserting to Qdrant"):
    client.upsert(collection_name=COLLECTION, points=points[start : start + UPSERT_BATCH])

info = client.get_collection(COLLECTION)
print(f"Collection '{COLLECTION}': {info.points_count} vectors | dim={EMBED_DIM}")

Collection 'news_chunks': 18990 vectors | dim=1024


### 2.4 Retriever

The retriever embeds the query (using the model's `query` prompt) and returns the top-k most similar chunks. Below are three example queries with k = 4.

In [24]:
def retrieve(query: str, k: int = 4) -> list[dict]:
    """Return the top-k most relevant chunks for a natural-language query."""
    query_vec = model.encode(query, prompt_name="query", normalize_embeddings=True).tolist()
    # query_points replaces the deprecated client.search() in qdrant-client >= 1.9
    response = client.query_points(
        collection_name=COLLECTION,
        query=query_vec,
        limit=k,
    )
    return [
        {
            "rank":       rank + 1,
            "score":      round(hit.score, 4),
            "chunk_id":   hit.payload["chunk_id"],
            "article_id": hit.payload["article_id"],
            "title":      hit.payload["title"],
            "source":     hit.payload["source"],
            "topic":      hit.payload["topic"],
            "text":       hit.payload["text"],
        }
        for rank, hit in enumerate(response.points)
    ]

def show_results(query, k=4):
    print(f"\nQuery: {query!r}\n" + "-" * 70)
    for r in retrieve(query, k):
        print(f"  [{r['rank']}] score={r['score']}  {r['source']} | {r['title'][:55]}")
        print(f"       {r['text'][:130]}\n")

# -- Three example queries --
show_results("What are the latest developments in artificial intelligence?")
show_results("How is climate change affecting global weather patterns?")
show_results("What is the current state of the US economy and inflation?")


Query: 'What are the latest developments in artificial intelligence?'
----------------------------------------------------------------------
  [1] score=0.659  The Guardian | Forget the AI job apocalypse. AI’s real threat is worke
       There are now major plans to expand AI skills across the workforce.

  [2] score=0.6454  BBC News | How the Trump-Xi summit could set superpower relations 
       This dynamic is playing itself out in the race to develop AI systems, perhaps the key new technological development of modern time

  [3] score=0.644  BBC News | How the Trump-Xi summit could set superpower relations 
       It is investing heavily in AI and humanoid robots.

  [4] score=0.6355  The Guardian | Forget the AI job apocalypse. AI’s real threat is worke
       Take Britain, which likes to present itself as being ambitious about AI.


Query: 'How is climate change affecting global weather patterns?'
----------------------------------------------------------------------
  [1] score

## 3. Generation

### 3.1 No-retrieval baseline

The generator calls a local LLM through the **Ollama** API, and the model is a parameter. First, the three queries are sent **without** any retrieved context, to see what the model says from its own training data alone. These answers are saved for comparison.

In [ ]:
import requests

DEFAULT_MODEL = "gemma4:latest"
OLLAMA_URL    = "http://localhost:11434/api/chat"

def generate(prompt: str, model: str = DEFAULT_MODEL, system: str = "") -> str:

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    resp = requests.post(
        OLLAMA_URL,
        json={"model": model, "messages": messages, "stream": False},
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()["message"]["content"]


# Baseline queries -- model answers from its own training knowledge only, no RAG corpus.
# These responses are saved to baseline_responses for comparison after the full RAG
# pipeline is wired up in section 3.4.
BASELINE_QUERIES = [
    "What are the latest developments in artificial intelligence?",
    "How is climate change affecting global weather patterns?",
    "What is the current state of the US economy and inflation?",
]

baseline_responses = {}
for q in BASELINE_QUERIES:
    print(f"\nQuery: {q!r}")
    print("-" * 70)
    answer = generate(q)
    baseline_responses[q] = answer
    print(answer[:600], "..." if len(answer) > 600 else "")


Query: 'What are the latest developments in artificial intelligence?'
----------------------------------------------------------------------
As an open-weights Large Language Model, I can provide an overview of the major and rapidly evolving trends in artificial intelligence. Because AI is advancing so quickly, "latest developments" usually refer to several parallel shifts in capability, efficiency, and governance.

Here is a comprehensive look at the current landscape of AI development, categorized by the area of impact:

***

## 🌐 I. Capability Breakthroughs (The "What AI Can Do")

The focus is moving away from simply generating coherent text toward generating complex, multi-format content and performing structured reasoning.

## ...

Query: 'How is climate change affecting global weather patterns?'
----------------------------------------------------------------------
Climate change does not simply mean that every day will be warmer; it means that the *variability*, *intensity*, an

### 3.2 Grounded RAG prompt

Following RAG prompting best practices, the system prompt tells the model to answer **only** from the provided passages, to say explicitly when the information is missing, and to cite the passage ID and source article title inline.

In [26]:

RAG_SYSTEM_PROMPT = """
You are a retrieval-augmented question-answering assistant.

Your task is to answer the user’s question using ONLY the information contained in the provided passages.

Instructions:
1. Base your answer strictly on the provided passages.
2. Do not use outside knowledge, assumptions, or unstated context.
3. If the passages do not contain enough information to answer the question, say so clearly.
4. Do not invent facts, sources, citations, dates, numbers, or explanations.
5. When making a claim, cite the relevant passage ID and source article title inline.
6. Use citations in this format:
   [Passage ID: <id>, Source: "<article title>"]
7. If multiple passages support the same claim, cite all relevant passages.
8. If passages conflict, explain the conflict and cite the passages on each side.
9. Keep the answer concise, accurate, and directly responsive to the user’s question.
10. If the user asks for information not covered by the passages, state:
   “The provided passages do not contain enough information to answer that.”

Input format:

Question:
{{user_question}}

Passages:
{{retrieved_passages}}

Each passage will include:
- Passage ID
- Source article title
- Passage text

Answer format:
- Direct answer first.
- Supporting details with inline citations.
- Missing information, if any.

Now answer the question using only the provided passages.
""" 


def build_rag_prompt(query: str, chunks: list[dict]) -> str:
    """Assemble the user-turn message from the query and retrieved chunks."""
    passages = "\n\n".join(
        f"[Passage {c['rank']} | chunk_id={c['chunk_id']} | {c['title']!r}]\n{c['text']}"
        for c in chunks
    )
    return f"Passages:\n{passages}\n\nQuestion: {query}"

### 3.3 Guardrails

Three post-generation checks are run on every answer:
1. **Citation presence**: the answer must cite at least one retrieved `chunk_id`.
2. **Citation validity**: every cited `chunk_id` must be in the retrieved set (no invented sources).
3. **Contradiction check**: the answer can't claim the information is missing *and* give a confident factual answer at the same time.

In [27]:
import re

def guardrail_1(answer: str, chunks: list[dict]) -> tuple[bool, str]:
    """Citation presence: the answer must reference at least one chunk_id."""
    chunk_ids = {c["chunk_id"] for c in chunks}
    cited = [cid for cid in chunk_ids if cid in answer]
    if not cited:
        return False, "Answer contains no chunk_id citations from the retrieved passages."
    return True, ""


def guardrail_2(answer: str, chunks: list[dict]) -> tuple[bool, str]:
    """Citation validity: every chunk_id cited in the answer must exist in the retrieved set."""
    valid_ids = {c["chunk_id"] for c in chunks}
    # chunk_ids have the form "123_45" (article_id underscore sentence_index)
    cited = set(re.findall(r'\b\d+_\d+\b', answer))
    phantom = cited - valid_ids
    if phantom:
        return False, f"Answer cites chunk_id(s) not in retrieved set: {phantom}"
    return True, ""


def guardrail_3(answer: str, chunks: list[dict]) -> tuple[bool, str]:
    """Unsupported-answer: answer must not simultaneously claim missing info and give a confident answer."""
    lower = answer.lower()

    missing_signals = [
        "not mentioned", "not provided", "not available", "no information",
        "cannot find", "cannot answer", "not in the passages", "not found",
        "insufficient information", "the passages do not", "passages don't",
        "i don't have", "not discussed",
    ]
    confident_signals = [
        "according to", "the answer is", "states that", "reports that",
        "confirms that", "reveals that", "indicates that", "it is clear",
        "the article", "the passage",
    ]

    has_missing   = any(p in lower for p in missing_signals)
    has_confident = any(p in lower for p in confident_signals)

    if has_missing and has_confident:
        return False, (
            "Answer simultaneously claims information is missing and provides "
            "a confident factual response -- contradictory."
        )
    return True, ""


GUARDRAILS = [guardrail_1, guardrail_2, guardrail_3]


def run_guardrails(answer: str, chunks: list[dict]) -> tuple[bool, list[str]]:
    """Run all guardrails; return (all_passed, list of failure reasons)."""
    failures = []
    for fn in GUARDRAILS:
        passed, reason = fn(answer, chunks)
        if not passed:
            failures.append(f"{fn.__name__}: {reason}")
    return len(failures) == 0, failures

### 3.4 End-to-end RAG flow

Retrieval and generation are connected into a single flow (retrieve → build prompt → generate → guardrails) and tested on the same three queries as the no-retrieval baseline.

In [28]:
def rag_answer(query: str, k: int = 4, model: str = DEFAULT_MODEL) -> dict:
    """Full RAG pipeline: retrieve -> build prompt -> generate -> guardrails."""
    chunks      = retrieve(query, k=k)
    user_prompt = build_rag_prompt(query, chunks)
    answer      = generate(user_prompt, model=model, system=RAG_SYSTEM_PROMPT)
    passed, failures = run_guardrails(answer, chunks)
    return {
        "query":              query,
        "answer":             answer,
        "chunks":             chunks,
        "guardrails_passed":  passed,
        "guardrail_failures": failures,
    }


# Test with the same three queries used in the no-RAG baseline above.
# Compare these answers to baseline_responses to see how grounding changes the output.
for q in BASELINE_QUERIES:
    print(f"\nQuery: {q!r}")
    print("=" * 70)
    result = rag_answer(q)
    print(result["answer"][:800], "..." if len(result["answer"]) > 800 else "")

    if result["guardrail_failures"]:
        print("\n[GUARDRAIL FAILURES]")
        for failure in result["guardrail_failures"]:
            print(f"  - {failure}")
    else:
        print("\n[All guardrails passed]")

    print("\nSources:")
    for c in result["chunks"]:
        print(
            f"  [{c['rank']}] score={c['score']}  "
            f"chunk_id={c['chunk_id']}  {c['source']} | {c['title'][:50]}"
        )


Query: 'What are the latest developments in artificial intelligence?'
Major developments and trends related to artificial intelligence include:

*   **Skills and Work:** There are major plans to expand AI skills across the workforce [Passage 1: chunk_id=65_37, Source: "Forget the AI job apocalypse. AI’s real threat is worker control and surveillance"].
*   **Global Race and Investment:** The development of AI systems is fueling a race, considered perhaps the key new technological development of modern times [Passage 2: chunk_id=81_72, Source: "How the Trump-Xi summit could set superpower relations for many years to come"]. This development is characterized by heavy investment in both AI and humanoid robots [Passage 3: chunk_id=81_67, Source: "How the Trump-Xi summit could set superpower relations for many years to come"].
*   **National Ambition:** Britain i ...

[All guardrails passed]

Sources:
  [1] score=0.659  chunk_id=65_37  The Guardian | Forget the AI job apocalypse. AI’s real

### 3.5 Chatbot UI

The same pipeline is served as a chat interface in Streamlit ([`streamlit_app.py`](streamlit_app.py)). It has model selection, a configurable k, an article browser, and the retrieved passages and guardrail status shown under each answer.

In [ ]:
import subprocess, sys

proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "streamlit_app.py",
     "--server.headless", "true"],
    cwd=".",
)
print(f"Streamlit started (PID {proc.pid})")
print("Open http://localhost:8501 in your browser.")
print("Re-run this cell to restart; kernel restart also stops the app.")

Streamlit started (PID 80408)
Open http://localhost:8501 in your browser.
Re-run this cell to restart; kernel restart also stops the app.


Screenshots of the app:

![COP30 question](assets/chat_cop30.png)

![Musk/OpenAI trial question](assets/chat_musk_openai.png)

![Airfare question](assets/chat_airfares.png)

## 4. Faithfulness evaluation

The corpus is unlabeled, so there are no ground-truth answers. Faithfulness (is every claim in the answer supported by the retrieved context?) only needs the question, the generated answer and the retrieved chunks.

### 4.1 Gold questions

Five articles were picked from the corpus, and for each one a factual question was written that can only be answered from that article.

In [33]:
article_facts_1 = {"question":"What screen size were announced for the new MacBook Pro with M5 chips?", 
                   "answer": "The new MacBook Pro with M5 chips features a 14-inch or 16-inch screen, depending on the model.",
                   "source": 15} #this is the article_id of the source article in the Qdrant collection that supports this fact. It's used for testing retrieval and citation in the RAG pipeline.
article_facts_2 = {"question":"What percentage of this year's wheat crop in Kansasis in poor condition according to the US Department of Agriculture?",
                   "answer": "Approximately 44% of this year's wheat crop in Kansas is in poor condition according to the US Department of Agriculture.",
                   "source": 28}
article_facts_3 = {"question":"Where did the COP30 climate summit take place?",
                   "answer": "The COP30 climate summit took place in Brazil.",
                   "source": 1}
articles_facts_4 = {"question":"What measure has been taken by Japan's biggest snack maker in response to the ink ingredient shortages?",
                   "answer": "The company has been forced to use black and white packaging for their products.",
                   "source": 42}
articles_facts_5 = {"question":"What will be new price of the Nintendo Switch 2 starting September 1, 2026?",
                   "answer": "The price of the Nintendo Switch 2 will be $499.99 starting September 1, 2026.",
                   "source": 494}

### 4.2 Run the baseline pipeline on the gold questions

Each question goes through the baseline RAG pipeline, and the (question, generated answer, retrieved chunks) tuple is collected.

In [34]:
gold_pairs = [
    article_facts_1,
    article_facts_2,
    article_facts_3,
    articles_facts_4,
    articles_facts_5,
]

rag_results = []
for i, pair in enumerate(gold_pairs, 1):
    print(f"\n[Q{i}] {pair['question']}")
    print("-" * 70)
    result = rag_answer(pair["question"], k=4)
    entry = {
        "question":           pair["question"],
        "gold_answer":        pair["answer"],
        "source_article_id":  pair["source"],
        "generated_answer":   result["answer"],
        "chunks":             result["chunks"],
        "guardrails_passed":  result["guardrails_passed"],
        "guardrail_failures": result["guardrail_failures"],
    }
    rag_results.append(entry)

    truncated = result["answer"][:300] + ("..." if len(result["answer"]) > 300 else "")
    print(f"Answer: {truncated}")
    gr_status = "PASSED" if result["guardrails_passed"] else f"FAILED — {result['guardrail_failures']}"
    print(f"Guardrails: {gr_status}")
    print(f"Chunks retrieved: {len(result['chunks'])}")
    for c in result["chunks"]:
        print(f"  [{c['rank']}] chunk_id={c['chunk_id']}  score={c['score']}  {c['source']} | {c['title'][:50]}")


[Q1] What screen size were announced for the new MacBook Pro with M5 chips?
----------------------------------------------------------------------
Answer: The provided passages indicate that the MacBook Pro M5 is available in sizes including the 14.2-inch screen [Passage 1: chunk_id=15_16, Source: 'MacBook Pro M5 review: serious power, still long battery life'] and the 14-inch MacBook Pro M5 [Passage 3: chunk_id=15_33, Source: 'MacBook Pro M5 review: ...
Guardrails: PASSED
Chunks retrieved: 4
  [1] chunk_id=15_16  score=0.6772  The Guardian | MacBook Pro M5 review: serious power, still long b
  [2] chunk_id=15_4  score=0.6597  The Guardian | MacBook Pro M5 review: serious power, still long b
  [3] chunk_id=15_33  score=0.6505  The Guardian | MacBook Pro M5 review: serious power, still long b
  [4] chunk_id=15_43  score=0.6424  The Guardian | MacBook Pro M5 review: serious power, still long b

[Q2] What percentage of this year's wheat crop in Kansasis in poor condition according to the

From the previous output we can see that some questions were correctly answered thanks to RAG, however, not all questions got the correct answer. For example, the first one that talks about the different screen sizes for the new MacBook. My hypothesis is that it was due to the chunking and the information being spread out between different sentences, so I will try a different chunking technique for the next section.

### 4.3 Baseline faithfulness (DeepEval)

The **DeepEval** `FaithfulnessMetric` extracts the claims in the answer and checks each one against the retrieval context. The judge LLM is the same local Ollama model, wrapped in a custom `DeepEvalBaseLLM` class.

In [38]:
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.metrics import FaithfulnessMetric
from deepeval.test_case import LLMTestCase


class OllamaJudge(DeepEvalBaseLLM):
    """Wraps the local Ollama API so DeepEval can use it as the LLM judge."""

    def __init__(self, model_name=DEFAULT_MODEL):
        # Set _model_name before super().__init__() because load_model() is called there
        self._model_name = model_name
        super().__init__(model=model_name)

    def load_model(self):
        return self._model_name

    def generate(self, prompt: str, schema=None) -> str:
        import requests as _req
        resp = _req.post(
            OLLAMA_URL,
            json={
                "model":    self._model_name,
                "messages": [{"role": "user", "content": prompt}],
                "stream":   False,
            },
            timeout=300,
        )
        resp.raise_for_status()
        return resp.json()["message"]["content"]

    async def a_generate(self, prompt: str, schema=None) -> str:
        return self.generate(prompt, schema)

    def get_model_name(self) -> str:
        return self._model_name


judge  = OllamaJudge(model_name=DEFAULT_MODEL)
metric = FaithfulnessMetric(
    model=judge,
    threshold=0.5,
    async_mode=False,   # use synchronous path to avoid event-loop conflicts in notebooks
    verbose_mode=True,
)

scores = []
for i, r in enumerate(rag_results, 1):
    tc = LLMTestCase(
        input=r["question"],
        actual_output=r["generated_answer"],
        retrieval_context=[c["text"] for c in r["chunks"]],
    )
    metric.measure(tc)
    score = metric.score
    scores.append(score)
    print(f"\nQ{i}: {r['question'][:65]}")
    print(f"  Faithfulness : {score:.3f}  ({'PASS' if metric.success else 'FAIL'})")
    if metric.reason:
        print(f"  Reason       : {metric.reason[:220]}")

mean_faithfulness = sum(scores) / len(scores)
print(f"\n{'=' * 60}")
print(f"Individual scores : {[round(s, 3) for s in scores]}")
print(f"Baseline mean     : {mean_faithfulness:.3f}")

**************************************************

Faithfulness Verbose Logs

**************************************************

Truths (limit=None):
[
    "The screen of the device is a 14.2in mini LED display with a resolution of 3024x1964 and 254 ppi.",
    "The screen supports ProMotion up to 120Hz.",
    "The device uses the macOS 26.4 Tahoe operating system.",
    "The MacBook Pro offers multiple configurations including processors (Apple M5, Pro, or Max), RAM (ranging up 
to 128GB), and storage (ranging up to 8TB SSD).",
    "The dimensions of the device are 221.2 x 312.6 x 15.5mm, and its weight ranges from 1.55 to 1.62kg.",
    "The M5 configuration supports wifi 6E, Bluetooth 5.3, and 3x Thunderbolt 4/USB 4 ports.",
    "The M5 Pro/Max configuration supports wifi 7, Bluetooth 6, Thread, and 3x Thunderbolt 5/USB 4 ports.",
    "All models listed include a Centre Stage camera and an SDXC slot.",
    "The starting prices are specified: M5 models start at £1,699; M5 Pro models start at £2,199; and M5 Max models
start at £3,599."
] 
 
Claims:
[
    "The provided passages indicate that the MacBook Pro M5 is available in sizes including the 14.2-inch screen 
[Passage 1: chunk_id=15_16, Source: 'MacBook Pro M5 review: serious power, still long battery life'] and the 
14-inch MacBook Pro M5 [Passage 3: chunk_id=15_33, Source: 'MacBook Pro M5 review: serious power, still long 
battery life']",
    "Additionally, one passage states that the latest MacBook Pro comes in two screen sizes [Passage 2: 
chunk_id=15_4, Source: 'MacBook Pro M5 review: serious power, still long battery life']."
] 
 
Verdicts:
[
    {
        "verdict": "idk",
        "reason": "The claim references external passages and sources (e.g., [Passage 1: chunk_id=15_16, Source: 
'MacBook Pro M5 review: serious power, still long battery life']) that are not provided in the retrieval context, 
making it impossible to verify the claim's assertion regarding available sizes."
    },
    {
        "verdict": "no",
        "reason": "The retrieval context explicitly states that the screen of the device is a 14.2in mini LED 
display, which contradicts the claim that the laptop comes in two screen sizes."
    }
]
 
Score: 0.5
Reason: The score is 0.50 because the actual output contradicts the retrieval context by claiming that the laptop 
comes in two screen sizes, when the retrieval context explicitly states that the screen of the device is a 14.2in 
mini LED display.

======================================================================

...

Faithfulness Verbose Logs

**************************************************

Truths (limit=None):
[
    "Nintendo announced that the Switch 2's MSRP will increase to $499.99 on September 1.",
    "The new MSRP of the Switch 2 represents a $50 (and approximately 11 percent) increase from its initial $449.99
launch price.",
    "Nintendo released updated financial forecasts today, projecting sales of 16.5 million Switch 2 units for the 
current fiscal year (ending March 2027).",
    "Nintendo cited 'the impact of these price revisions' when projecting the 16.5 million Switch 2 unit sales.",
    "A lawsuit references price increases for Nintendo Switch 2 accessories that were announced in April 2025.",
    "After reviewing the Switch 2 following its launch last year, a warning was issued that interested customers 
should buy early due to potential launch price increases."
] 
 
Claims:
[
    "The provided passages do not contain enough information to answer that."
] 
 
Verdicts:
[
    {
        "verdict": "no",
        "reason": "The context contains extensive details, including the specific MSRP increase to $499.99 on 
September 1, the calculation of the $50 (11%) increase from the initial $449.99 launch price, and projected sales 
of 16.5 million units."
    }
]
 
Score: 0.0
Reason: The score is 0.00 because the actual output fails to incorporate crucial details, such as the specific MSRP
increase to $499.99 on September 1, the calculation of the $50 (11%) increase from the initial $449.99 launch 
price, or the projected sales of 16.5 million units, which are extensively detailed in the retrieval context.

======================================================================


Q5: What will be new price of the Nintendo Switch 2 starting Septembe
  Faithfulness : 0.000  (FAIL)
  Reason       : The score is 0.00 because the actual output fails to incorporate crucial details, such as the specific MSRP increase to $499.99 on September 1, the calculation of the $50 (11%) increase from the initial $449.99 launch pr

Individual scores : [0.5, 1.0, 1.0, 1.0, 0.0]
Baseline mean     : 0.700


As we can see the first question gets a score of 0.5 while the last one gets a 0. All other questions passed with a score of 1. 

## 5. Improving the pipeline

The goal is to make at least two concrete changes that could improve faithfulness, and implement them.

I think a more appropriate chunking technique would be very beneficial to improve RAG in this case, at least from what I hypothesize from the examples, for now I think I will switch from sentence to multi-sentence chunking. Another change would be to change the retrieval technique to something more advanced, I will try a Hybrid RAG approach that uses vector search and BM25 search as well to have semantic and keyword retrieval. 

In [39]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

COLLECTION_V2 = "news_chunks_v2"

# ── Improved chunking: adaptive 2-3 sentence windows ──────────────────
def chunk_article_v2(row, short_threshold=40):
    """Group sentences into 2-sentence chunks.
    If the combined word count is still below short_threshold, extend to 3 sentences.
    This gives richer context per chunk without losing granularity for long sentences.
    """
    sentences = [s for s in sent_tokenize(row["body"]) if len(s.split()) >= MIN_WORDS]
    chunks, idx, i = [], 0, 0
    while i < len(sentences):
        group = [sentences[i]]
        i += 1
        if i < len(sentences):                      # always add a 2nd sentence
            group.append(sentences[i]); i += 1
        if i < len(sentences) and len(" ".join(group).split()) < short_threshold:
            group.append(sentences[i]); i += 1     # extend to 3rd if still short
        chunks.append({
            "chunk_id":   f"{row['article_id']}_{idx}",
            "article_id": int(row["article_id"]),
            "title":      row["title"],
            "source":     row["source"],
            "topic":      row["topic"],
            "url":        row["url"],
            "text":       " ".join(group),
        })
        idx += 1
    return chunks

all_chunks_v2 = []
for _, row in df.iterrows():
    all_chunks_v2.extend(chunk_article_v2(row))

chunks_df_v2 = pd.DataFrame(all_chunks_v2)
print(f"V1 chunks : {len(chunks_df):>6}  avg {chunks_df['text'].str.split().str.len().mean():.0f} words/chunk")
print(f"V2 chunks : {len(chunks_df_v2):>6}  avg {chunks_df_v2['text'].str.split().str.len().mean():.0f} words/chunk")

# ── Embed v2 chunks ────────────────────────────────────────────────────
texts_v2    = chunks_df_v2["text"].tolist()
embeds_list = []
for i in tqdm(range(0, len(texts_v2), BATCH_SIZE), desc="Encoding v2 chunks"):
    embeds_list.append(
        model.encode(texts_v2[i : i + BATCH_SIZE], normalize_embeddings=True, show_progress_bar=False)
    )
embeddings_v2 = np.vstack(embeds_list)
print(f"Embeddings: {embeddings_v2.shape}")

# ── Upload to Qdrant (fresh collection) ───────────────────────────────
if client.collection_exists(COLLECTION_V2):
    client.delete_collection(COLLECTION_V2)
client.create_collection(
    collection_name=COLLECTION_V2,
    vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
)

points_v2 = [
    PointStruct(
        id      = i,
        vector  = embeddings_v2[i].tolist(),
        payload = {k: row[k] for k in ["chunk_id","article_id","title","source","topic","url","text"]},
    )
    for i, (_, row) in enumerate(chunks_df_v2.reset_index(drop=True).iterrows())
]
for start in tqdm(range(0, len(points_v2), UPSERT_BATCH), desc="Upserting to Qdrant"):
    client.upsert(collection_name=COLLECTION_V2, points=points_v2[start : start + UPSERT_BATCH])

info_v2 = client.get_collection(COLLECTION_V2)
print(f"Collection '{COLLECTION_V2}': {info_v2.points_count} vectors | dim={EMBED_DIM}")

V1 chunks :  18990  avg 23 words/chunk
V2 chunks :   8229  avg 54 words/chunk


Embeddings: (8229, 1024)


Collection 'news_chunks_v2': 8229 vectors | dim=1024


In [40]:
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# ── BM25 index over v2 chunks ──────────────────────────────────────────
tokenized_v2 = [text.lower().split() for text in texts_v2]
bm25_v2      = BM25Okapi(tokenized_v2)
print(f"BM25 index built on {len(tokenized_v2)} chunks")

# ── Cross-encoder reranker ─────────────────────────────────────────────
# ms-marco-MiniLM-L-6-v2: fast, lightweight passage reranker (~80 MB)
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder reranker loaded")


# ── Hybrid retrieval: vector + BM25 → RRF → cross-encoder reranking ───
def retrieve_v2(query: str, k: int = 4, fetch_k: int = 20) -> list[dict]:
    """
    1. Retrieve fetch_k candidates from Qdrant (dense vector search).
    2. Retrieve fetch_k candidates from BM25 (sparse keyword search).
    3. Merge with Reciprocal Rank Fusion (RRF, k_rrf=60).
    4. Rerank the top-fetch_k fused candidates with a cross-encoder.
    5. Return the top-k reranked results.
    """
    # 1. Dense vector search
    qvec  = model.encode(query, prompt_name="query", normalize_embeddings=True).tolist()
    vresp = client.query_points(collection_name=COLLECTION_V2, query=qvec, limit=fetch_k)
    vec_rank    = {hit.id: rank for rank, hit in enumerate(vresp.points)}
    vec_payload = {hit.id: hit.payload for hit in vresp.points}

    # 2. Sparse BM25 search (returns scores over the full corpus; take top fetch_k)
    bm25_scores = bm25_v2.get_scores(query.lower().split())
    bm25_top    = np.argsort(bm25_scores)[::-1][:fetch_k]
    bm25_rank   = {int(idx): rank for rank, idx in enumerate(bm25_top)}

    # 3. RRF fusion — score = 1/(k+rank_vec) + 1/(k+rank_bm25)
    K_RRF   = 60
    all_ids = set(vec_rank) | set(bm25_rank)
    rrf     = {
        cid: 1 / (K_RRF + vec_rank.get(cid, fetch_k)) + 1 / (K_RRF + bm25_rank.get(cid, fetch_k))
        for cid in all_ids
    }
    top_ids = sorted(rrf, key=rrf.__getitem__, reverse=True)[:fetch_k]

    # 4. Resolve payloads (Qdrant cache first, then DataFrame fallback)
    candidates = []
    for pt_id in top_ids:
        if pt_id in vec_payload:
            candidates.append(vec_payload[pt_id])
        else:
            row = chunks_df_v2.iloc[int(pt_id)]
            candidates.append(
                {k: row[k] for k in ["chunk_id","article_id","title","source","topic","url","text"]}
            )

    # 5. Cross-encoder reranking
    ce_scores = reranker.predict([(query, c["text"]) for c in candidates])
    reranked  = sorted(zip(ce_scores, candidates), reverse=True, key=lambda x: x[0])

    return [
        {
            "rank":       rank + 1,
            "score":      round(float(score), 4),
            "chunk_id":   c["chunk_id"],
            "article_id": c["article_id"],
            "title":      c["title"],
            "source":     c["source"],
            "topic":      c["topic"],
            "url":        c["url"],
            "text":       c["text"],
        }
        for rank, (score, c) in enumerate(reranked[:k])
    ]


def rag_answer_v2(query: str, k: int = 4, model_name: str = DEFAULT_MODEL) -> dict:
    """Improved RAG: 2-3-sentence chunks + hybrid retrieval + cross-encoder reranking."""
    chunks      = retrieve_v2(query, k=k)
    user_prompt = build_rag_prompt(query, chunks)
    answer      = generate(user_prompt, model=model_name, system=RAG_SYSTEM_PROMPT)
    passed, failures = run_guardrails(answer, chunks)
    return {"query": query, "answer": answer, "chunks": chunks,
            "guardrails_passed": passed, "guardrail_failures": failures}


# ── Quick smoke test ───────────────────────────────────────────────────
print("\nSmoke test (hybrid retrieval):")
for r in retrieve_v2("What are the latest developments in artificial intelligence?"):
    print(f"  [{r['rank']}] score={r['score']:>7.4f}  {r['source']} | {r['title'][:50]}")

BM25 index built on 8229 chunks


BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-encoder reranker loaded

Smoke test (hybrid retrieval):
  [1] score=-3.2490  The Guardian | Forget the AI job apocalypse. AI’s real threat is 
  [2] score=-4.4223  BBC News | How the Trump-Xi summit could set superpower relat
  [3] score=-5.7377  BBC News | Should you really trust health advice from an AI c
  [4] score=-7.0064  The Guardian | Molière Ex Machina: AI used to create ‘new work’ b


### 5.1 Re-evaluation: baseline vs. improved

Same 5 gold questions, same judge, now with the improved pipeline.

In [41]:
# ── Run improved RAG on the same 5 gold questions ─────────────────────
rag_results_v2 = []
for i, pair in enumerate(gold_pairs, 1):
    print(f"\n[Q{i}] {pair['question']}")
    print("-" * 70)
    result = rag_answer_v2(pair["question"], k=4)
    rag_results_v2.append({
        "question":           pair["question"],
        "gold_answer":        pair["answer"],
        "source_article_id":  pair["source"],
        "generated_answer":   result["answer"],
        "chunks":             result["chunks"],
        "guardrails_passed":  result["guardrails_passed"],
        "guardrail_failures": result["guardrail_failures"],
    })
    truncated = result["answer"][:300] + ("..." if len(result["answer"]) > 300 else "")
    print(f"Answer: {truncated}")
    gr_status = "PASSED" if result["guardrails_passed"] else f"FAILED — {result['guardrail_failures']}"
    print(f"Guardrails: {gr_status}")
    print(f"Top chunk : {result['chunks'][0]['source']} | {result['chunks'][0]['title'][:50]}")

# ── DeepEval faithfulness on v2 results ───────────────────────────────
print("\n" + "=" * 60)
print("DeepEval Faithfulness — Improved Pipeline")
print("=" * 60)

scores_v2 = []
for i, r in enumerate(rag_results_v2, 1):
    tc = LLMTestCase(
        input=r["question"],
        actual_output=r["generated_answer"],
        retrieval_context=[c["text"] for c in r["chunks"]],
    )
    metric.measure(tc)
    score = metric.score
    scores_v2.append(score)
    print(f"\nQ{i}: {r['question'][:65]}")
    print(f"  Faithfulness : {score:.3f}  ({'PASS' if metric.success else 'FAIL'})")
    if metric.reason:
        print(f"  Reason       : {metric.reason[:220]}")

mean_v2 = sum(scores_v2) / len(scores_v2)

# ── Side-by-side comparison table ─────────────────────────────────────
print(f"\n{'=' * 68}")
print(f"{'Q':<4} {'Question':<46} {'Base':>6}  {'V2':>6}  {'Δ':>6}")
print("-" * 68)
for i, (s_base, s_v2, r) in enumerate(zip(scores, scores_v2, rag_results_v2), 1):
    delta = s_v2 - s_base
    arrow = "↑" if delta > 0 else ("↓" if delta < 0 else "=")
    print(f"Q{i:<3} {r['question'][:46]:<46} {s_base:>6.3f}  {s_v2:>6.3f}  {delta:>+5.3f} {arrow}")
print("-" * 68)
print(f"{'Mean':<51} {mean_faithfulness:>6.3f}  {mean_v2:>6.3f}  {mean_v2 - mean_faithfulness:>+5.3f}")


[Q1] What screen size were announced for the new MacBook Pro with M5 chips?
----------------------------------------------------------------------
Answer: The MacBook Pro is available in two screen sizes [Passage 1: chunk_id=15_1, Source: "MacBook Pro M5 review: serious power, still long battery life"].

Specific versions and screen sizes mentioned include:
*   The 14in version [Passage 1: chunk_id=15_1, Source: "MacBook Pro M5 review: serious power, ...
Guardrails: PASSED
Top chunk : The Guardian | MacBook Pro M5 review: serious power, still long b

[Q2] What percentage of this year's wheat crop in Kansasis in poor condition according to the US Department of Agriculture?
----------------------------------------------------------------------
Answer: According to the weekly crop condition report issued by an arm of the US Department of Agriculture, 44% of Kansas's wheat is rated in very poor to poor condition [Passage 1: chunk_id=28_6, Source: ‘The worst time for wheat’: US farmers fac

Answer: The provided passages do not contain enough information to answer that.

While the passages mention an increase in the Switch 2’s MSRP to $499.99 starting on September 1 [Passage 1: chunk\_id=494\_1, Source: "The Nintendo Switch 2 is getting more expensive later this year"], none of the passages spe...
Guardrails: FAILED — ['guardrail_1: Answer contains no chunk_id citations from the retrieved passages.']
Top chunk : Ars Technica | The Nintendo Switch 2 is getting more expensive la

DeepEval Faithfulness — Improved Pipeline


**************************************************

Faithfulness Verbose Logs

**************************************************

...

Faithfulness Verbose Logs

**************************************************

Truths (limit=None):
[
    "Nintendo announced the Switch 2’s MSRP will increase to $499.99 on September 1.",
    "The increase represents a $50 (or about 11 percent) rise from the original $449.99 launch price.",
    "Nintendo cited 'changes in market conditions' and 'the global business outlook' as reasons for the impending 
price increase.",
    "A lawsuit reported price increases for Nintendo Switch 2 accessories were announced in April 2025, with 
increases ranging from $1 to $10 per product.",
    "The lawsuit specified that the Nintendo Switch 2 Pro Controller price increased from $79.99 to $84.99, and the
Nintendo Switch 2 Dock Set price increased from $109.99 to $119.99.",
    "The lawsuit also mentioned increases for the original Switch console in August 2025, which ranged from $30 to 
$50 depending on the model.",
    "The lawsuit stated that Nintendo President Shuntaro Furukawa told investors in May 2025 that any imposed 
tariffs would be recognized as a cost and incorporated into the price."
] 
 
Claims:
[
    "The provided passages do not contain enough information to answer that.",
    "The passages mention an increase in the Switch 2’s MSRP to $499.99 starting on September 1 [Passage 1: 
chunk_id=494_1, Source: "The Nintendo Switch 2 is getting more expensive later this year"]",
    "None of the passages specify that this increase occurs in the year 2026."
] 
 
Verdicts:
[
    {
        "verdict": "no",
        "reason": "The context provides multiple specific facts, such as the MSRP increasing to $499.99 on 
September 1, and price increases for accessories in April 2025, showing that the passages contain sufficient 
information to answer questions."
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    }
]
 
Score: 0.6666666666666666
Reason: The score is 0.67 because while the context provides multiple specific facts, such as the MSRP increasing 
to $499.99 on September 1 and price increases for accessories in April 2025, the actual output did not utilize or 
fully incorporate all of these detailed points.

======================================================================


Q5: What will be new price of the Nintendo Switch 2 starting Septembe
  Faithfulness : 0.667  (PASS)
  Reason       : The score is 0.67 because while the context provides multiple specific facts, such as the MSRP increasing to $499.99 on September 1 and price increases for accessories in April 2025, the actual output did not utilize or 

Q    Question                                         Base      V2       Δ
--------------------------------------------------------------------
Q1   What screen size were announced for the new Ma  0.500   1.000  +0.500 ↑
Q2   What percentage of this year's wheat crop in K  1.000   1.000  +0.000 =
Q3   Where did the COP30 climate summit take place?  1.000   1.000  +0.000 =
Q4   What measure has been taken by Japan's biggest  1.000   1.000  +0.000 =
Q5   What will be new price of the Nintendo Switch   0.000   0.667  +0.667 ↑
--------------------------------------------------------------------
Mean                                                 0.700   

## 6. Takeaways: retrieval quality vs. generation faithfulness

With this project I saw the impact retrieval quality can have on generation faithfulness. In my initial version where the ChatBot relied exclusively on the vector database created with simple strategies like sentence level chunking and normal retrieval the responses sometimes fell short when comparing to the ground truth in the test. This suggested the model was not always getting the most relevant or complete context before generating its response. After modifying the chunking strategy and adopting a hybrid retrieval approach that combined BM25 with vector based search, the ChatBot was able to generate more faithful answers, as we can see from the increase in performance from 0.70 to 0.93. In conclusion, the experiment showed that answer quality can be improved with the right retrieval system. 